# 🚂 Notebook 1 — Training Pipeline (`train.py`)

**What this file does:** It is the "conductor" that orchestrates the entire training process. It calls feature_extractor, coreset, and memory_bank in the right order.

**Why it exists:** Instead of running 3 separate notebooks manually, this ties everything into a single `main()` call that processes all categories automatically.

**Important:** PatchCore has NO gradient-based training. "Training" means:
1. Extract patch features from all good images
2. Compress them with coreset sampling
3. Save the result to disk

**How it fits:** This is the entry point. It uses all three `core/` modules.

## 1 — Setup
Mount Drive, install dependencies, unzip data, and configure logging.
This notebook is **fully self-contained** — it includes the feature
extractor, coreset, and memory bank code inline so you can run
the entire training pipeline from a single notebook.

In [ ]:
# ---- Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Install ----
!pip install -q faiss-cpu tqdm

# ---- Unzip datasets ----
import zipfile, glob, os, logging

DRIVE_DATA = '/content/drive/MyDrive/defects-detection-CV'
MVTEC_ROOT = '/content/data'
os.makedirs(MVTEC_ROOT, exist_ok=True)

for zf in glob.glob(os.path.join(DRIVE_DATA, '*.zip')):
    name = os.path.basename(zf).split('-')[0]
    dest = os.path.join(MVTEC_ROOT, name)
    if os.path.isdir(dest):
        print(f'{name} already unzipped.')
        continue
    print(f'Unzipping {name}...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(MVTEC_ROOT)
    print(f'  done.')

# ---- Logging ----
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('train')
logger.info('Setup complete.')

## 2 — Configuration
All tuneable values in one place. Change once, used everywhere.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
import faiss
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from typing import Dict, Tuple
from tqdm import tqdm

# ---- Config ----
CATEGORIES     = ['bottle', 'carpet', 'screw']
DATA_ROOT      = '/content/data'
OUTPUT_DIR     = '/content/drive/MyDrive/defects-detection/outputs'
IMAGE_SIZE     = 224
BATCH_SIZE     = 32
NUM_WORKERS    = 2
BACKBONE_NAME  = 'wide_resnet50_2'
CORESET_RATIO  = 0.01          # keep 1% of patches
FEAT_CKPT_EVERY  = 10          # feature extraction checkpoint interval
CORE_CKPT_EVERY  = 500         # coreset checkpoint interval
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

logger.info('Device: %s | Categories: %s | Coreset: %.1f%%',
            DEVICE, CATEGORIES, CORESET_RATIO * 100)

## 3 — Inline Dependencies
The next three cells paste in the code from `feature_extractor.py`,
`coreset.py`, and `memory_bank.py` so this notebook runs standalone.

### 3a — Feature Extractor functions
Extracts layer2+layer3 features, applies local smoothing, aligns,
flattens, and collects patches for every image.

In [ ]:
def extract_layer_features(
    model: nn.Module, image_batch: torch.Tensor,
    hook_dict: Dict[str, torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Forward pass → return layer2 & layer3 feature maps."""
    with torch.no_grad():
        _ = model(image_batch)
    return hook_dict['layer2'], hook_dict['layer3']


def locally_aware_patches(
    feature_map: torch.Tensor, patch_size: int = 3,
) -> torch.Tensor:
    """Average-pool each position with its neighbours for local context."""
    return F.avg_pool2d(feature_map, kernel_size=patch_size,
                        stride=1, padding=patch_size // 2)


def align_and_concat(
    feat_l2: torch.Tensor, feat_l3: torch.Tensor,
) -> torch.Tensor:
    """Upsample layer3 to layer2 resolution and concatenate channels."""
    h, w = feat_l2.shape[2], feat_l2.shape[3]
    feat_l3_up = F.interpolate(feat_l3, size=(h, w),
                               mode='bilinear', align_corners=False)
    return torch.cat([feat_l2, feat_l3_up], dim=1)


def flatten_patches(
    feature_map: torch.Tensor,
) -> Tuple[torch.Tensor, Tuple[int, int]]:
    """Reshape (B,C,H,W) → (B*H*W, C) and return spatial shape."""
    B, C, H, W = feature_map.shape
    flat = feature_map.permute(0, 2, 3, 1).reshape(-1, C)
    return flat, (H, W)


def extract_dataset_features(
    model: nn.Module, dataloader: DataLoader,
    hook_dict: Dict[str, torch.Tensor], device: str,
    checkpoint_path: str = '', checkpoint_every: int = 10,
) -> torch.Tensor:
    """Extract locally-aware patch features for every training image."""
    all_features: list = []
    start_batch: int = 0

    if checkpoint_path:
        try:
            ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
            all_features = ckpt['features']
            start_batch = ckpt['next_batch']
            logger.info('Resumed features at batch %d', start_batch)
        except FileNotFoundError:
            logger.info('No feature checkpoint — starting fresh.')
        except Exception as exc:
            logger.warning('Feature checkpoint load failed (%s)', exc)

    batches = list(dataloader)
    for batch_idx, images in enumerate(
        tqdm(batches[start_batch:], desc='Extracting features',
             initial=start_batch, total=len(batches)),
        start=start_batch,
    ):
        images = images.to(device)
        fl2, fl3 = extract_layer_features(model, images, hook_dict)
        fl2 = locally_aware_patches(fl2)
        fl3 = locally_aware_patches(fl3)
        combined = align_and_concat(fl2, fl3)
        flat, _ = flatten_patches(combined)
        all_features.append(flat.cpu())

        if checkpoint_path and (batch_idx + 1) % checkpoint_every == 0:
            try:
                torch.save({'features': all_features,
                            'next_batch': batch_idx + 1}, checkpoint_path)
                logger.info('Feature checkpoint at batch %d', batch_idx + 1)
            except Exception as exc:
                logger.warning('Feature checkpoint failed: %s', exc)

    result = torch.cat(all_features, dim=0)

    if checkpoint_path and os.path.exists(checkpoint_path):
        try: os.remove(checkpoint_path)
        except: pass

    logger.info('Features extracted: %s', result.shape)
    return result

### 3b — Coreset functions
Greedy farthest-point sampling to compress ~200k patches down to ~2k.

In [ ]:
def greedy_coreset_sampling(
    features: torch.Tensor, ratio: float, device: str,
    checkpoint_path: str = '', checkpoint_every: int = 500,
) -> torch.Tensor:
    """Farthest-point sampling to select the most diverse subset."""
    n = features.shape[0]
    n_select = max(1, int(n * ratio))
    logger.info('Coreset: %d / %d (%.1f%%)', n_select, n, ratio * 100)

    features_gpu = features.to(device)
    selected_idx: list = []
    min_distances = torch.full((n,), float('inf'), device=device)

    if checkpoint_path:
        try:
            ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
            selected_idx = ckpt['selected_idx']
            min_distances = ckpt['min_distances'].to(device)
            logger.info('Resumed coreset at %d', len(selected_idx))
        except FileNotFoundError:
            pass
        except Exception as exc:
            logger.warning('Coreset checkpoint failed (%s)', exc)

    if len(selected_idx) == 0:
        seed = torch.randint(0, n, (1,)).item()
        selected_idx.append(seed)
        diff = features_gpu - features_gpu[seed].unsqueeze(0)
        min_distances = torch.sum(diff ** 2, dim=1)

    remaining = n_select - len(selected_idx)
    pbar = tqdm(range(remaining), desc='Coreset sampling',
                initial=len(selected_idx), total=n_select)
    for i in pbar:
        next_idx = torch.argmax(min_distances).item()
        selected_idx.append(next_idx)
        diff = features_gpu - features_gpu[next_idx].unsqueeze(0)
        min_distances = torch.minimum(min_distances, torch.sum(diff ** 2, dim=1))

        if checkpoint_path and (i + 1) % checkpoint_every == 0:
            try:
                torch.save({'selected_idx': selected_idx,
                            'min_distances': min_distances.cpu()}, checkpoint_path)
                logger.info('Coreset checkpoint (%d/%d)', len(selected_idx), n_select)
            except Exception as exc:
                logger.warning('Coreset checkpoint failed: %s', exc)
    pbar.close()

    if checkpoint_path and os.path.exists(checkpoint_path):
        try: os.remove(checkpoint_path)
        except: pass

    return torch.tensor(selected_idx, dtype=torch.long)


def subsample_memory_bank(
    features: torch.Tensor, ratio: float, device: str,
    checkpoint_path: str = '', checkpoint_every: int = 500,
) -> torch.Tensor:
    """Wrapper: coreset sample and return the reduced tensor."""
    indices = greedy_coreset_sampling(features, ratio, device,
                                      checkpoint_path, checkpoint_every)
    reduced = features[indices].cpu()
    logger.info('Subsampled: %s → %s', features.shape, reduced.shape)
    return reduced

### 3c — Memory Bank functions
Save/load the final compressed features to/from Google Drive.

In [ ]:
def save_memory_bank(memory_bank: torch.Tensor, save_path: str) -> None:
    """Save memory bank tensor to disk."""
    try:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(memory_bank, save_path)
        logger.info('Saved memory bank %s → %s', memory_bank.shape, save_path)
    except Exception as exc:
        logger.error('Save failed: %s', exc)
        raise

---
# 🚂 `train.py` — Implementation
Now we define the 3 training functions that tie everything together.

### Function 1 — `build_memory_bank`
The core training logic in two steps:
1. **Extract** all patch features from the training images.
2. **Compress** them with coreset subsampling.

Both steps have checkpoints so you don’t lose progress on Colab disconnects.

In [ ]:
def build_memory_bank(
    model: nn.Module,
    dataloader: DataLoader,
    hook_dict: Dict[str, torch.Tensor],
    coreset_ratio: float,
    device: str,
    category: str = '',
) -> torch.Tensor:
    """Orchestrate feature extraction + coreset subsampling.

    Args:
        model: Frozen backbone on device.
        dataloader: Training DataLoader (good images only).
        hook_dict: Hook dict attached to the backbone.
        coreset_ratio: Fraction of patches to keep.
        device: 'cuda' or 'cpu'.
        category: Category name for checkpoint naming.

    Returns:
        Memory bank tensor (n_select, D) on CPU.
    """
    # Step 1: Extract features
    logger.info('Step 1/2: Extracting features...')
    feat_ckpt = os.path.join(OUTPUT_DIR, 'raw_features', f'{category}_feat_ckpt.pt')
    os.makedirs(os.path.dirname(feat_ckpt), exist_ok=True)

    all_features = extract_dataset_features(
        model, dataloader, hook_dict, device,
        checkpoint_path=feat_ckpt,
        checkpoint_every=FEAT_CKPT_EVERY,
    )
    logger.info('Raw features shape: %s', all_features.shape)

    # Step 2: Coreset subsample
    logger.info('Step 2/2: Coreset subsampling...')
    core_ckpt = os.path.join(OUTPUT_DIR, 'memory_banks', f'{category}_coreset_ckpt.pt')
    os.makedirs(os.path.dirname(core_ckpt), exist_ok=True)

    memory_bank = subsample_memory_bank(
        all_features, coreset_ratio, device,
        checkpoint_path=core_ckpt,
        checkpoint_every=CORE_CKPT_EVERY,
    )
    logger.info('Memory bank shape: %s', memory_bank.shape)
    return memory_bank

### Function 2 — `build_and_save_bank`
The full pipeline for **one category**:
1. Load the backbone with hooks
2. Create the training DataLoader
3. Call `build_memory_bank`
4. Save the result to Google Drive

Skips the category if a memory bank already exists.

In [ ]:
def build_and_save_bank(category: str) -> None:
    """Full training pipeline for a single category.

    Args:
        category: MVTec category name (e.g. 'bottle').
    """
    logger.info('========== Category: %s ==========', category)

    save_path = os.path.join(OUTPUT_DIR, 'memory_banks', f'{category}_memory_bank.pt')
    if os.path.exists(save_path):
        logger.info('[%s] Already done — skipping.', category)
        return

    # ---- Backbone ----
    model = getattr(models, BACKBONE_NAME)(weights='IMAGENET1K_V1')
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False

    hook_dict: Dict[str, torch.Tensor] = {}
    def _hook(name):
        def fn(m, i, o): hook_dict[name] = o.detach()
        return fn
    model.layer2.register_forward_hook(_hook('layer2'))
    model.layer3.register_forward_hook(_hook('layer3'))

    # ---- Dataset ----
    class _DS(Dataset):
        def __init__(self):
            d = os.path.join(DATA_ROOT, category, 'train', 'good')
            self.paths = sorted([os.path.join(d, f) for f in os.listdir(d)
                                  if f.lower().endswith(('.png','.jpg','.jpeg'))])
            self.tfm = T.Compose([
                T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ])
            logger.info('[%s] %d training images', category, len(self.paths))
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            return self.tfm(Image.open(self.paths[i]).convert('RGB'))

    dl = DataLoader(_DS(), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

    # ---- Build & Save ----
    mb = build_memory_bank(model, dl, hook_dict, CORESET_RATIO, DEVICE, category)
    save_memory_bank(mb, save_path)
    logger.info('[%s] ✅ Complete!', category)

### Function 3 — `main`
Simply loops over all categories and calls `build_and_save_bank`.
This is the only function you need to call.

In [ ]:
def main() -> None:
    """Train PatchCore for every configured category."""
    logger.info('PatchCore Training Pipeline')
    logger.info('Categories: %s', CATEGORIES)
    logger.info('Device: %s | Coreset: %.1f%%', DEVICE, CORESET_RATIO * 100)

    for category in CATEGORIES:
        build_and_save_bank(category)

    logger.info('🎉 All categories complete!')

---
## ▶️ Run Training
One cell to train everything. Already-completed categories are skipped.

In [ ]:
main()

## ✅ Summary
After running, Google Drive should contain:
```
defects-detection/outputs/
  memory_banks/
    bottle_memory_bank.pt   ← (~1.6k patches, 1536 dims)
    carpet_memory_bank.pt   ← (~2.2k patches, 1536 dims)
    screw_memory_bank.pt    ← (~2.5k patches, 1536 dims)
```

**Next step:** The `test.py` notebook will load these memory banks,
build FAISS indices, score test images, and generate heatmaps.